In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt

# Full reproducibility
torch.manual_seed(42)
np.random.seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# ────────────────────────────────────────────────────────────────
# Model (unchanged from strong run)
# ────────────────────────────────────────────────────────────────

class GoldstoneHyperHoloBlock(nn.Module):
    def __init__(self, dim=72, depth=5, alpha=1.7, slip_factor=0.12, ng_strength=0.09):
        super().__init__()
        self.depth = depth
        self.alpha = alpha
        self.proj = nn.Linear(dim, dim)
        self.slip_gen = nn.Linear(dim, dim)
        self.holo_proj = nn.Linear(dim * 2, dim * 4)
        self.inv_holo = nn.Linear(dim * 4, dim)
        self.ng_proj = nn.Linear(dim, dim)
        self.ng_strength = ng_strength
        self.slip_factor = slip_factor
        self.norm_h = nn.LayerNorm(dim)
        self.norm_out = nn.LayerNorm(dim)

    def forward(self, x, level=0, t_step=0.0, training=False):
        if level >= self.depth:
            return x

        slip = torch.tanh(self.slip_gen(x)) * self.slip_factor
        h = torch.relu(self.proj(x + slip))
        h = self.norm_h(h)

        h_child = self.forward(h, level + 1, t_step, training)

        concat = torch.cat([h, h_child], dim=-1)
        boundary = self.holo_proj(concat)
        resolved = self.inv_holo(boundary)
        fb = resolved * h

        if training:
            vev_rand = torch.tanh(torch.randn_like(h.mean(dim=-1, keepdim=True)))
            phase_noise = torch.randn_like(h) * 0.15
            vev_scale = 0.005
            phase_scale = 1.0
        else:
            vev_rand = torch.randn_like(h.mean(dim=-1, keepdim=True)) * 0.3
            phase_noise = torch.randn_like(h) * 0.08
            vev_scale = 0.0025
            phase_scale = 0.6

        vev = torch.norm(h, dim=-1, keepdim=True) * vev_scale * torch.tanh(vev_rand)
        h_broken = h + vev

        proj = self.ng_proj(h_broken)
        norm_hb_sq = h_broken.norm(dim=-1, keepdim=True)**2 + 1e-8
        dot = (proj * h_broken).sum(-1, keepdim=True) / norm_hb_sq
        goldstone = proj - dot * h_broken

        phase = torch.sin(t_step * 2.0 * np.pi + phase_noise * phase_scale)
        h = h + self.ng_strength * goldstone * phase
        h = self.norm_h(h)

        out = x + fb + 0.04 * h_child
        if training:
            mask_prob = max(0.1, 1.0 / ((level + 1) ** self.alpha))
            mask = (torch.rand_like(out) < mask_prob).float()
            out = out * mask
        out = self.norm_out(out)

        return out


class CoherenceNet(nn.Module):
    def __init__(self, input_dim, hidden_dim=72):
        super().__init__()
        self.embed = nn.Linear(input_dim, hidden_dim)
        self.fractal_core = GoldstoneHyperHoloBlock(hidden_dim)
        self.head = nn.Linear(hidden_dim, input_dim)

    def forward(self, x, steps=1, training=False):
        h = self.embed(x)
        for t in range(steps):
            h = self.fractal_core(h, t_step=t / max(1, steps), training=training)
        return self.head(h)


# ────────────────────────────────────────────────────────────────
# Helpers (unchanged)
# ────────────────────────────────────────────────────────────────

def o4_energy(spins, J=1.0):
    energy = 0.0
    N = spins.shape[0]
    for i in range(N):
        for j in range(N):
            energy += np.dot(spins[i, j], spins[i, (j + 1) % N])
            energy += np.dot(spins[i, j], spins[(i + 1) % N, j])
    return -J * energy


def get_observables(spins):
    en = o4_energy(spins)
    mag = spins.mean(axis=(0, 1))
    mag2 = np.dot(mag, mag)
    return en, mag2


def metropolis_step(spins, beta):
    N = spins.shape[0]
    i, j = np.random.randint(0, N, 2)
    old_s = spins[i, j].copy()
    delta = np.random.randn(4) * 0.3
    new_s = old_s + delta
    norm = np.linalg.norm(new_s)
    if norm > 1e-8:
        new_s /= norm
    else:
        new_s = np.random.randn(4)
        new_s /= np.linalg.norm(new_s) + 1e-8

    dE = 0.0
    neigh = [(i, (j-1)%N), (i, (j+1)%N), ((i-1)%N, j), ((i+1)%N, j)]
    for ni, nj in neigh:
        dE += np.dot(new_s - old_s, spins[ni, nj])
    dE = -dE

    if dE < 0 or np.random.rand() < np.exp(-beta * dE):
        spins[i, j] = new_s
    return spins


def autocorrelation(ts, lag):
    if len(ts) <= lag:
        return 0.0
    mean = np.mean(ts)
    var = np.var(ts)
    if var < 1e-12:
        return 0.0
    cov = np.mean((ts[:-lag] - mean) * (ts[lag:] - mean))
    return cov / var


def integrated_autocorr_time(obs, max_lag=200, cutoff=0.04):
    if len(obs) < 30:
        return 1.0
    ac = [autocorrelation(obs, lag) for lag in range(1, min(max_lag, len(obs)) + 1)]
    for i, val in enumerate(ac):
        if val < cutoff:
            ac = ac[:i+1]
            break
    tau = 1 + 2 * np.sum(ac)
    return max(tau, 1.0)


# ────────────────────────────────────────────────────────────────
# Parameters — exact match to 7.6× mag² config (no scheduler)
# ────────────────────────────────────────────────────────────────

N = 4
input_dim = N * N * 4
beta = 1.1
mcmc_steps = 25000
burnin = 6000
n_thermal_configs = 1000
n_random_fraction = 0.25

mix_factor = 0.68
proposal_noise = 0.022


def initialize_o4_lattice(N):
    spins = np.random.randn(N, N, 4)
    norms = np.linalg.norm(spins, axis=-1, keepdims=True)
    spins /= np.maximum(norms, 1e-8)
    return spins


# ────────────────────────────────────────────────────────────────
# Training data
# ────────────────────────────────────────────────────────────────

print("Generating thermal configurations...")
thermal_configs = []
spins_eq = initialize_o4_lattice(N)

for step in range(mcmc_steps + burnin):
    spins_eq = metropolis_step(spins_eq, beta)
    if step >= burnin and (step - burnin) % 8 == 0:
        thermal_configs.append(spins_eq.copy())
        if len(thermal_configs) >= n_thermal_configs:
            break

print(f"Generated {len(thermal_configs)} thermal configs")

n_random = int(n_thermal_configs * n_random_fraction)
random_configs = [initialize_o4_lattice(N) for _ in range(n_random)]

all_training_spins = thermal_configs + random_configs
np.random.shuffle(all_training_spins)

configs = [torch.tensor(s.flatten(), dtype=torch.float32).unsqueeze(0) for s in all_training_spins]
print(f"Total training examples: {len(configs)}\n")


# ────────────────────────────────────────────────────────────────
# Train — NO scheduler this time (to match strong mag² run)
# ────────────────────────────────────────────────────────────────

model = CoherenceNet(input_dim=input_dim)
optimizer = optim.Adam(model.parameters(), lr=0.0006, weight_decay=3e-6)
criterion = nn.MSELoss()

n_epochs = 120
print_interval = 10

print("Training...")
for epoch in range(n_epochs):
    total_loss = 0.0
    for config in configs:
        noisy_target = config + torch.randn_like(config) * 0.09
        noisy_target = noisy_target / (torch.norm(noisy_target, dim=-1, keepdim=True) + 1e-8)

        pred = model(config, steps=1, training=True)
        loss_noise = criterion(pred, noisy_target)
        loss_recon = criterion(pred, config)
        loss = 0.85 * loss_noise + 0.15 * loss_recon

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(configs)
    if (epoch + 1) % print_interval == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}: Avg loss {avg_loss:.6f}")

print("Training finished.\n")


# ────────────────────────────────────────────────────────────────
# Proposal — 2 micro-steps, 0.65 mix
# ────────────────────────────────────────────────────────────────

def neural_proposal(spins, model, beta):
    try:
        current = spins.astype(np.float32).copy()
        for _ in range(2):
            flat = torch.from_numpy(current.flatten()).unsqueeze(0)
            with torch.no_grad():
                pred_flat = model(flat, steps=1, training=False).numpy().flatten()
            pred = pred_flat.reshape(N, N, 4)
            pred /= np.maximum(np.linalg.norm(pred, axis=-1, keepdims=True), 1e-8)
            current += 0.65 * (pred - current)

        proposed = current + np.random.randn(*current.shape) * proposal_noise
        proposed /= np.maximum(np.linalg.norm(proposed, axis=-1, keepdims=True), 1e-8)

        dE = o4_energy(proposed) - o4_energy(spins)

        if dE < 0:
            accept = True
        else:
            arg = -beta * dE
            arg = np.clip(arg, -700, 700)
            accept = np.random.rand() < np.exp(arg)

        return proposed.copy() if accept else spins.copy(), accept

    except Exception as e:
        print(f"Proposal failed: {e} → fallback")
        return metropolis_step(spins.copy(), beta), False


# ────────────────────────────────────────────────────────────────
# Run chains
# ────────────────────────────────────────────────────────────────

print("Running standard Metropolis chain...")
np.random.seed(100)
torch.manual_seed(100)
spins_std = initialize_o4_lattice(N)
energy_std, mag2_std = [], []

for step in range(mcmc_steps + burnin):
    spins_std = metropolis_step(spins_std, beta)
    if step >= burnin:
        en, m2 = get_observables(spins_std)
        energy_std.append(en)
        mag2_std.append(m2)

print("Running kernel-accelerated chain...")
np.random.seed(101)
torch.manual_seed(101)
spins_neural = initialize_o4_lattice(N)
energy_neural, mag2_neural = [], []
accepts = total_prop = 0

for step in range(mcmc_steps + burnin):
    proposed, accepted = neural_proposal(spins_neural, model, beta)
    spins_neural = proposed
    total_prop += 1
    if accepted:
        accepts += 1
    if step >= burnin:
        en, m2 = get_observables(spins_neural)
        energy_neural.append(en)
        mag2_neural.append(m2)

accept_rate = accepts / total_prop if total_prop > 0 else 0
print(f"Neural proposal acceptance rate: {accept_rate:.3%}\n")


# ────────────────────────────────────────────────────────────────
# Results
# ────────────────────────────────────────────────────────────────

tau_en_std = integrated_autocorr_time(energy_std)
tau_en_k   = integrated_autocorr_time(energy_neural)
tau_m2_std = integrated_autocorr_time(mag2_std)
tau_m2_k   = integrated_autocorr_time(mag2_neural)

print("Energy:")
print(f"  τ std     : {tau_en_std:.2f}")
print(f"  τ kernel  : {tau_en_k:.2f}")
print(f"  gain      : {tau_en_std / tau_en_k:.2f}x\n")

print("Magnetization²:")
print(f"  τ std     : {tau_m2_std:.2f}")
print(f"  τ kernel  : {tau_m2_k:.2f}")
print(f"  gain      : {tau_m2_std / tau_m2_k:.2f}x\n")

Generating thermal configurations...
Generated 1000 thermal configs
Total training examples: 1250

Training...
Epoch   1: Avg loss 0.062640
Epoch  10: Avg loss 0.019664
Epoch  20: Avg loss 0.018956
Epoch  30: Avg loss 0.018878
Epoch  40: Avg loss 0.018811
Epoch  50: Avg loss 0.018769
Epoch  60: Avg loss 0.018740
Epoch  70: Avg loss 0.018732
Epoch  80: Avg loss 0.018727
Epoch  90: Avg loss 0.018720
Epoch 100: Avg loss 0.018728
Epoch 110: Avg loss 0.018721
Epoch 120: Avg loss 0.018720
Training finished.

Running standard Metropolis chain...
Running kernel-accelerated chain...
Neural proposal acceptance rate: 93.339%

Energy:
  τ std     : 266.97
  τ kernel  : 45.71
  gain      : 5.84x

Magnetization²:
  τ std     : 299.40
  τ kernel  : 58.62
  gain      : 5.11x

